# Training speed debug

Diagnoses why one training batch of `Ocean_Emulator_UNet.ipynb` takes ~10 s.

**Run on the GPU node, top to bottom, then save the notebook** (outputs are read back from the saved file).

Sections:
1. Setup -- executes the definition cells of `Ocean_Emulator_UNet.ipynb` (matched by marker text, not index), so this notebook always uses the same model/loss code.
2. Hardware / backend flags.
3. Scaling with rollout length (`n_steps` = 1, 2, 4, 12).
4. Data path cost (collate + host-to-device copy) vs GPU-resident batches.
5. Speed-up variants: `cudnn.benchmark`, zero vs replicate padding, bf16 autocast.
6. `torch.profiler` op breakdown for one training step.
7. Summary.

In [ ]:
# -----------------------------------------------------------------------------
# 1. Setup: execute only the definition cells from the main notebook.
# Skips PET pipeline construction, cache building and training.
# -----------------------------------------------------------------------------
import json
import time
from pathlib import Path

MAIN_NB = Path("Ocean_Emulator_UNet.ipynb")
if not MAIN_NB.exists():
    MAIN_NB = Path("/g/data/v46/txs156/OM2-emulator/notebooks/Ocean_Emulator_UNet.ipynb")

_main_cells = [
    "".join(c["source"])
    for c in json.load(open(MAIN_NB))["cells"]
    if c["cell_type"] == "code"
]

# Markers identify the cells to run, in order.
SETUP_MARKERS = [
    "Cell purpose: import dependencies",       # imports, device, paths
    "Notebook-wide time configuration",        # time_start, train_end, ...
    "mask, normalisation = build_normalisation",  # forcing_variables, mean, deviation
    "ACCESS_OHC_accessor = ACCESS_OHC(",       # accessor (used for area_t)
    "class ForwardUNet",                       # model wrapper
    "def rollout_loss(",                       # loss
    "class ForwardLightningWrapper",           # Lightning wrapper + unpack helpers
    "class RolloutTensorDataset",              # dataset + time lookup
]

N_ROLLOUT_STEPS = 12  # normally set in the PET pipeline cell, which is skipped here

for marker in SETUP_MARKERS:
    matches = [src for src in _main_cells if marker in src]
    if len(matches) != 1:
        raise RuntimeError(f"Marker {marker!r} matched {len(matches)} cells in {MAIN_NB}")
    t0 = time.perf_counter()
    exec(compile(matches[0], f"<main notebook: {marker}>", "exec"), globals())
    print(f"ran [{marker}] in {time.perf_counter() - t0:.1f}s")

# Model construction cell (needs N_ROLLOUT_STEPS and the accessor above).
_model_cells = [src for src in _main_cells if "base_model = ForwardUNet(" in src]
assert len(_model_cells) == 1, len(_model_cells)
exec(compile(_model_cells[0], "<main notebook: model>", "exec"), globals())
print("model built; CLOSURE_WEIGHT =", CLOSURE_WEIGHT, "| n_steps =", lightning_model.n_steps)

In [ ]:
# -----------------------------------------------------------------------------
# Load the cached rollout tensors (same files the main notebook trains on).
# -----------------------------------------------------------------------------
t0 = time.perf_counter()
train_cache = torch.load(database + "/OM2-emulator/data/train_rollout_12step.pt", map_location="cpu")
print(f"loaded train cache in {time.perf_counter() - t0:.1f}s")

for k, v in train_cache.items():
    print(f"  {k:22s} {str(tuple(v.shape)):28s} {v.dtype}  {v.numel() * v.element_size() / 1e9:.2f} GB")

train_dataset = RolloutTensorDataset(
    train_cache["prior"],
    train_cache["posterior"],
    train_cache["forcing"],
    train_cache["target_time_indices"],
    train_cache["initial_forcing"],  # needed by the closure loss; regenerate old caches
)
BATCH_SIZE = 32
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
print("batches per epoch:", len(train_dl))


def to_device(x, device):
    if torch.is_tensor(x):
        return x.to(device, non_blocking=True)
    if isinstance(x, (tuple, list)):
        return type(x)(to_device(xi, device) for xi in x)
    return x


def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()


def timed(fn, warmup=1, iters=3):
    """Mean wall time of fn() in seconds, synchronising the GPU."""
    for _ in range(warmup):
        fn()
    sync()
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    sync()
    return (time.perf_counter() - t0) / iters


lightning_model.to(device).train()
cpu_batch = next(iter(train_dl))
gpu_batch = to_device(cpu_batch, device)
sync()

RESULTS = {}

In [ ]:
# -----------------------------------------------------------------------------
# 2. Hardware and backend flags.
# -----------------------------------------------------------------------------
import os
import subprocess

print("torch", torch.__version__, "| lightning", L.__version__)
print("device used by notebook:", device)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, f"| {props.total_memory / 1e9:.1f} GB | sm_{props.major}{props.minor}")
    print("bf16 supported:", torch.cuda.is_bf16_supported())
print("cudnn.benchmark:", torch.backends.cudnn.benchmark)
print("cudnn.allow_tf32:", torch.backends.cudnn.allow_tf32)
print("cuda.matmul.allow_tf32:", torch.backends.cuda.matmul.allow_tf32)
print("float32_matmul_precision:", torch.get_float32_matmul_precision())

n_params = sum(p.numel() for p in base_model.parameters())
print(f"model parameters: {n_params:,}")
for name, m in base_model.named_modules():
    if isinstance(m, PartialConv2d):
        c = m.conv
        print(f"  {name:22s} {c.in_channels:3d}->{c.out_channels:3d} k={c.kernel_size} s={c.stride} pad_mode={c.padding_mode}")

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    apps = subprocess.run(
        ["nvidia-smi", "--query-compute-apps=pid,used_memory", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip().splitlines()
    others = [a for a in apps if a.split(",")[0].strip() != str(os.getpid())]
    if others:
        warnings.warn(
            "OTHER PROCESSES ARE USING THIS GPU -- timings will be contaminated. "
            f"Shut down other kernels first: {others}"
        )
    else:
        print("GPU check: no other compute processes on this GPU.")
except FileNotFoundError:
    print("nvidia-smi not found")

In [ ]:
# -----------------------------------------------------------------------------
# 3. Cost vs rollout length: forward+backward of one batch (B=32), no optimiser.
# Expect roughly linear scaling in n_steps if the model is the bottleneck.
# -----------------------------------------------------------------------------

def train_step(n_steps, batch=None):
    lightning_model.n_steps = n_steps
    lightning_model.zero_grad(set_to_none=True)
    loss = lightning_model._step(gpu_batch if batch is None else batch)
    loss.backward()
    return loss


# Forward only, single UNet call, to get a per-pass cost.
prior0 = ForwardLightningWrapper._squeeze_variable_axis(gpu_batch[0][0])
forcing0 = gpu_batch[1][:, 0]
with torch.no_grad():
    t_fwd = timed(lambda: base_model(prior0, forcing0, lightning_model.mask), warmup=2, iters=5)
print(f"single UNet forward, B={BATCH_SIZE}: {t_fwd * 1e3:.1f} ms")
RESULTS["unet_forward_B32_ms"] = t_fwd * 1e3

for n in [1, 2, 4, 12]:
    torch.cuda.reset_peak_memory_stats() if device.type == "cuda" else None
    try:
        t = timed(lambda: train_step(n), warmup=1, iters=2)
    except torch.cuda.OutOfMemoryError as err:
        print(f"n_steps={n}: OOM ({err})")
        torch.cuda.empty_cache()
        continue
    peak = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else float("nan")
    print(f"n_steps={n:2d}: {t:6.2f} s/batch | {t / n * 1e3:7.1f} ms per step | peak mem {peak:.1f} GB")
    RESULTS[f"fwd_bwd_nsteps{n}_s"] = t
    RESULTS[f"peak_mem_nsteps{n}_GB"] = peak

lightning_model.n_steps = N_ROLLOUT_STEPS

In [ ]:
# -----------------------------------------------------------------------------
# 4. Data path: how long does fetching + copying a batch take, vs the compute?
# -----------------------------------------------------------------------------

def time_loader(dl, max_batches=5):
    it = iter(dl)
    t_collate, t_h2d = 0.0, 0.0
    n = 0
    for _ in range(max_batches):
        t0 = time.perf_counter()
        try:
            b = next(it)
        except StopIteration:
            break
        t1 = time.perf_counter()
        to_device(b, device)
        sync()
        t2 = time.perf_counter()
        t_collate += t1 - t0
        t_h2d += t2 - t1
        n += 1
    return t_collate / n, t_h2d / n


batch_bytes = sum(
    t.numel() * t.element_size()
    for t in [cpu_batch[0][0], cpu_batch[0][1], cpu_batch[1], cpu_batch[2]]
)
print(f"bytes per batch: {batch_bytes / 1e9:.2f} GB")

c, h = time_loader(train_dl)
print(f"default loader : collate {c:.3f} s | host->device {h:.3f} s")
RESULTS["loader_collate_s"], RESULTS["loader_h2d_s"] = c, h

pinned_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
c, h = time_loader(pinned_dl)
print(f"pinned loader  : collate+pin {c:.3f} s | host->device {h:.3f} s")
RESULTS["pinned_collate_s"], RESULTS["pinned_h2d_s"] = c, h
del pinned_dl

In [ ]:
# -----------------------------------------------------------------------------
# 5. Speed-up variants, each timed at n_steps=2 (fwd+bwd) against a baseline.
# bf16 timings are for speed only: the closure term needs pred_t cast to float32
# before it is used for real training.
# -----------------------------------------------------------------------------
import contextlib

VARIANT_STEPS = 2
partial_convs = [m for m in base_model.modules() if isinstance(m, PartialConv2d)]


@contextlib.contextmanager
def padding_mode(mode):
    old = [(m.conv.padding_mode, m.mask_conv.padding_mode) for m in partial_convs]
    for m in partial_convs:
        m.conv.padding_mode = mode
        m.mask_conv.padding_mode = mode
    try:
        yield
    finally:
        for m, (a, b) in zip(partial_convs, old):
            m.conv.padding_mode, m.mask_conv.padding_mode = a, b


@contextlib.contextmanager
def cudnn_benchmark(flag):
    old = torch.backends.cudnn.benchmark
    torch.backends.cudnn.benchmark = flag
    try:
        yield
    finally:
        torch.backends.cudnn.benchmark = old


# V100 (sm_70) has fp16 tensor cores but no bf16 hardware; bf16 is only
# worth testing on sm_80+.
amp_dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16


def run_variant(name, *contexts):
    with contextlib.ExitStack() as stack:
        for ctx in contexts:
            stack.enter_context(ctx)
        try:
            t = timed(lambda: train_step(VARIANT_STEPS), warmup=2, iters=3)
        except Exception as err:  # report and continue with the other variants
            print(f"{name:32s} FAILED: {type(err).__name__}: {err}")
            return
    RESULTS[f"variant[{name}]_s"] = t
    base = RESULTS.get("variant[baseline]_s", t)
    print(f"{name:32s} {t:6.3f} s  ({base / t:4.2f}x vs baseline)")


run_variant("baseline")
run_variant("cudnn.benchmark", cudnn_benchmark(True))
run_variant("zeros padding", padding_mode("zeros"))
run_variant("circular padding", padding_mode("circular"))
run_variant(f"autocast {amp_dtype}", torch.autocast("cuda", dtype=amp_dtype))
run_variant("benchmark + zeros", cudnn_benchmark(True), padding_mode("zeros"))
run_variant(
    f"autocast {amp_dtype} + benchmark + zeros",
    torch.autocast("cuda", dtype=amp_dtype), cudnn_benchmark(True), padding_mode("zeros"),
)

# Full 12-step batch: baseline vs fp32 fixes vs fp32 fixes + AMP.
for name, ctxs in [
    ("12-step baseline", []),
    ("12-step benchmark + zeros", [cudnn_benchmark(True), padding_mode("zeros")]),
    (f"12-step {amp_dtype} + benchmark + zeros",
     [torch.autocast("cuda", dtype=amp_dtype), cudnn_benchmark(True), padding_mode("zeros")]),
]:
    with contextlib.ExitStack() as stack:
        for ctx in ctxs:
            stack.enter_context(ctx)
        torch.cuda.reset_peak_memory_stats()
        try:
            t = timed(lambda: train_step(N_ROLLOUT_STEPS), warmup=1, iters=2)
        except torch.cuda.OutOfMemoryError as err:
            print(f"{name:40s} OOM: {err}")
            torch.cuda.empty_cache()
            continue
    RESULTS[f"{name}_s"] = t
    print(f"{name:40s} {t:6.2f} s/batch | peak mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

# Closure-loss cost: rerun baseline with closure disabled.
_old_cw = lightning_model.closure_weight
lightning_model.closure_weight = 0.0
t = timed(lambda: train_step(VARIANT_STEPS), warmup=1, iters=3)
lightning_model.closure_weight = _old_cw
RESULTS["variant[closure off]_s"] = t
print(f"{'closure off (actual)':32s} {t:6.3f} s  ({RESULTS['variant[baseline]_s'] / t:4.2f}x vs baseline)")

In [ ]:
# -----------------------------------------------------------------------------
# 6. Profiler: op-level breakdown of one fwd+bwd at n_steps=2 (baseline settings).
# -----------------------------------------------------------------------------
from torch.profiler import profile, ProfilerActivity

train_step(2)  # warm-up
sync()
activities = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device.type == "cuda" else [])
t0 = time.perf_counter()
with profile(activities=activities) as prof:
    train_step(2)
    sync()
wall = time.perf_counter() - t0

events = prof.key_averages()
sort_key = "self_cuda_time_total"
if events and not hasattr(events[0], "self_cuda_time_total"):
    sort_key = "self_device_time_total"
print(events.table(sort_by=sort_key, row_limit=25))

from torch.autograd import DeviceType
gpu_us = sum(
    getattr(e, "self_device_time_total", getattr(e, "self_cuda_time_total", 0))
    for e in prof.events()
    if e.device_type == DeviceType.CUDA
)
print(f"wall {wall:.3f} s | summed GPU kernel time {gpu_us / 1e6:.3f} s | GPU busy ~{100 * gpu_us / 1e6 / wall:.0f}% (kernels only)")
RESULTS["profile_wall_s"] = wall
RESULTS["profile_gpu_busy_pct"] = 100 * gpu_us / 1e6 / wall

In [ ]:
# -----------------------------------------------------------------------------
# 7. Summary
# -----------------------------------------------------------------------------
for k, v in RESULTS.items():
    print(f"{k:50s} {v:10.3f}")